In [1]:
#import your modules here
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from collections import Counter
import csv


In [2]:
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

TAXONOMY_DIR = PROJECT_ROOT / "data" / "taxonomy"
TAXONOMY_DIR.mkdir(parents=True, exist_ok=True)

from utils.fetch_data import fetch_xml, extract_locs, parse_obiavi_geo_path

SITEMAP_INDEX_URL = "https://www.imot.bg/sitemap/index.xml"

HEADERS = {
    "User-Agent": "Mozilla/5.0 valuation-research-bot"
}

print(DATA_DIR)

D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_1\data


In [3]:
file_path_obiavi_urls = DATA_DIR / "imot_obiavi_urls.txt"
file_path_deal_types =  TAXONOMY_DIR / "valid_deal_types.csv"
file_path_property_types =  TAXONOMY_DIR / "valid_property_types.csv"
file_path_valid_geo_types = TAXONOMY_DIR / "valid_geo_paths.csv"

In [4]:
# 1. Read sitemap index
index_xml = fetch_xml(SITEMAP_INDEX_URL)

# 2. Extract child sitemap URLs
child_sitemaps = extract_locs(index_xml)

print("Child sitemaps found:", len(child_sitemaps))

# 3. Extract all URLs from all child sitemaps
all_urls = []

for sitemap_url in child_sitemaps:
    try:
        xml_text = fetch_xml(sitemap_url)
        urls = extract_locs(xml_text)

        all_urls.extend(urls)

        print(f"{sitemap_url}: {len(urls)} URLs")

    except Exception as exc:
        print(f"FAILED {sitemap_url}: {exc}")

print("Total URLs found:", len(all_urls))

# 4. Extract /obiavi/ category/search URLs
obiavi_urls = [
    url for url in all_urls
    if "https://www.imot.bg/obiavi/" in url
]

print("/obiavi/ URLs found:", len(obiavi_urls))

# 5. Optional: save them
with open(file_path_obiavi_urls, "w", encoding="utf-8") as f:
    for url in obiavi_urls:
        f.write(url + "\n")

print("Saved imot_obiavi_urls.txt")

Child sitemaps found: 44
https://www.imot.bg/sitemap/agencies-pages.xml.gz: 5248 URLs
https://www.imot.bg/sitemap/agentsii.xml.gz: 137 URLs
https://www.imot.bg/sitemap/listings-1.xml.gz: 49962 URLs
https://www.imot.bg/sitemap/listings-10.xml.gz: 49952 URLs
https://www.imot.bg/sitemap/listings-11.xml.gz: 49963 URLs
https://www.imot.bg/sitemap/listings-12.xml.gz: 49959 URLs
https://www.imot.bg/sitemap/listings-13.xml.gz: 49952 URLs
https://www.imot.bg/sitemap/listings-14.xml.gz: 49950 URLs
https://www.imot.bg/sitemap/listings-15.xml.gz: 49956 URLs
https://www.imot.bg/sitemap/listings-16.xml.gz: 49951 URLs
https://www.imot.bg/sitemap/listings-17.xml.gz: 49960 URLs
https://www.imot.bg/sitemap/listings-18.xml.gz: 49952 URLs
https://www.imot.bg/sitemap/listings-19.xml.gz: 49956 URLs
https://www.imot.bg/sitemap/listings-2.xml.gz: 49954 URLs
https://www.imot.bg/sitemap/listings-20.xml.gz: 49956 URLs
https://www.imot.bg/sitemap/listings-21.xml.gz: 49952 URLs
https://www.imot.bg/sitemap/listings

In [5]:
with open(file_path_obiavi_urls, "r", encoding="utf-8") as f:
    obiavi_urls = [line.strip() for line in f if line.strip()]

print("/obiavi/ URLs loaded:", len(obiavi_urls))
print("First 10 URLs:")

for url in obiavi_urls[:10]:
    print(url)


parsed_geo_routes = []

for url in obiavi_urls:
    parsed = parse_obiavi_geo_path(url)
    if parsed:
        parsed_geo_routes.append(parsed)

print("Input /obiavi/ URLs:", len(obiavi_urls))
print("Parsed routes:", len(parsed_geo_routes))


/obiavi/ URLs loaded: 27469
First 10 URLs:
https://www.imot.bg/obiavi/naemi/grad-blagoevgrad
https://www.imot.bg/obiavi/naemi/grad-blagoevgrad/alen-mak
https://www.imot.bg/obiavi/naemi/grad-blagoevgrad/alen-mak/dvustaen
https://www.imot.bg/obiavi/naemi/grad-blagoevgrad/dvustaen
https://www.imot.bg/obiavi/naemi/grad-blagoevgrad/alen-mak/sklad
https://www.imot.bg/obiavi/naemi/grad-blagoevgrad/sklad
https://www.imot.bg/obiavi/naemi/grad-blagoevgrad/varosha
https://www.imot.bg/obiavi/naemi/grad-blagoevgrad/varosha/ednostaen
https://www.imot.bg/obiavi/naemi/grad-blagoevgrad/ednostaen
https://www.imot.bg/obiavi/naemi/grad-blagoevgrad/vtora-promishlena-zona
Input /obiavi/ URLs: 27469
Parsed routes: 27426


In [6]:
parsed_geo_routes

[{'url': 'https://www.imot.bg/obiavi/naemi/grad-blagoevgrad',
  'deal_type': 'naemi',
  'geo_path': 'grad-blagoevgrad',
  'geo_level_count': 1,
  'geo_parts': ['grad-blagoevgrad'],
  'base_property_type': None},
 {'url': 'https://www.imot.bg/obiavi/naemi/grad-blagoevgrad/alen-mak',
  'deal_type': 'naemi',
  'geo_path': 'grad-blagoevgrad/alen-mak',
  'geo_level_count': 2,
  'geo_parts': ['grad-blagoevgrad', 'alen-mak'],
  'base_property_type': None},
 {'url': 'https://www.imot.bg/obiavi/naemi/grad-blagoevgrad/alen-mak/dvustaen',
  'deal_type': 'naemi',
  'geo_path': 'grad-blagoevgrad/alen-mak',
  'geo_level_count': 2,
  'geo_parts': ['grad-blagoevgrad', 'alen-mak'],
  'base_property_type': 'dvustaen'},
 {'url': 'https://www.imot.bg/obiavi/naemi/grad-blagoevgrad/dvustaen',
  'deal_type': 'naemi',
  'geo_path': 'grad-blagoevgrad',
  'geo_level_count': 1,
  'geo_parts': ['grad-blagoevgrad'],
  'base_property_type': 'dvustaen'},
 {'url': 'https://www.imot.bg/obiavi/naemi/grad-blagoevgrad/al

In [7]:
geo_counter = Counter(
    (r["deal_type"], r["geo_path"])
    for r in parsed_geo_routes
)

print("Unique geo paths:", len(geo_counter))

for (deal_type, geo_path), count in geo_counter.most_common(30):
    print(deal_type, geo_path, count)

Unique geo paths: 5549
naemi grad-burgas 21
naemi oblast-plovdiv 21
naemi grad-sofiya 21
prodazhbi oblast-blagoevgrad 21
prodazhbi oblast-blagoevgrad/gr-bansko 21
prodazhbi oblast-burgas 21
prodazhbi oblast-burgas/gr-pomorie 21
prodazhbi oblast-veliko-tarnovo 21
prodazhbi oblast-pazardzhik 21
prodazhbi oblast-pazardzhik/gr-velingrad 21
prodazhbi grad-plovdiv 21
prodazhbi oblast-plovdiv 21
prodazhbi grad-ruse 21
prodazhbi oblast-sofiya 21
naemi oblast-burgas 20
naemi grad-varna 20
naemi grad-ruse 20
prodazhbi grad-burgas 20
prodazhbi oblast-burgas/gr-sveti-vlas 20
prodazhbi grad-varna 20
prodazhbi grad-varna/tsentar 20
prodazhbi oblast-varna 20
prodazhbi grad-veliko-tarnovo 20
prodazhbi grad-veliko-tarnovo/tsentar 20
prodazhbi grad-vratsa 20
prodazhbi oblast-gabrovo 20
prodazhbi oblast-dobrich 20
prodazhbi grad-lovech 20
prodazhbi grad-pazardzhik 20
prodazhbi grad-plovdiv/tsentar 20


In [8]:
deal_types = [
    {
        "deal_type": "prodazhbi",
        "deal_type_en": "sale",
        "deal_type_bg": "Продажби"
    },
    {
        "deal_type": "naemi",
        "deal_type_en": "rent",
        "deal_type_bg": "Наеми"
    }
]

with open(file_path_deal_types, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["deal_type", "deal_type_en", "deal_type_bg"]
    )
    writer.writeheader()
    writer.writerows(deal_types)

print("Saved valid_deal_types.csv")

Saved valid_deal_types.csv


In [9]:
property_counter = Counter(
    r["base_property_type"]
    for r in parsed_geo_routes
    if r["base_property_type"]
)

with open(file_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "property_type",
        "route_count"
    ])

    for property_type, count in sorted(property_counter.items()):
        writer.writerow([
            property_type,
            count
        ])

print("Saved valid_property_types.csv")
print("Unique property types:", len(property_counter))

NameError: name 'file_path' is not defined

In [ ]:
geo_counter = Counter(
    (r["deal_type"], r["geo_path"])
    for r in parsed_geo_routes
)

with open(file_path_valid_geo_types, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "deal_type",
        "geo_path",
        "geo_level_count",
        "geo_1",
        "geo_2",
        "geo_3",
        "route_count"
    ])

    for (deal_type, geo_path), route_count in sorted(geo_counter.items()):
        parts = geo_path.split("/")
        padded = parts + ["", "", ""]

        writer.writerow([
            deal_type,
            geo_path,
            len(parts),
            padded[0],
            padded[1],
            padded[2],
            route_count
        ])

print("Saved valid_geo_paths.csv")
print("Unique geo paths:", len(geo_counter))